This second notebook solves for the ground-state of the He-like systems, by parameterizing the Slater determinant,
$$
\Psi(\boldsymbol{X}_1,\boldsymbol{X}_2,Z_{\rm{eff}}) =\left[ \psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,1}})\psi_{1\rm{s}}(\boldsymbol{r}_2,Z_{\rm{eff,2}}) +  \psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,1}})\psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,2}})\right] \left(\frac{1}{\sqrt{2}}\uparrow_1\downarrow_2 -  \frac{1}{\sqrt{2}}\downarrow_1\uparrow_2 \right)
$$

where the bracketed spin term is a normalized Singlet state, and can effectively be ignored for the calculation of Hamiltonian matrix element. $Z_{\rm{eff},1 (2)}$ paramterize each of the one-electron wavefunctions.

The Hamiltonian for a He-like system is,
$$
\mathcal{H} = -\frac{1}{2} \left(\nabla^2_1 + \nabla^2_2 \right) - \frac{Z}{r_1}- \frac{Z}{r_2} + \frac{1}{r_{12}},
$$
where $Z$ is the charge of the nucleus and not to be confused with $Z_{\rm{eff}}$. 

In [2]:
import sympy as sp

# Define symbolic variables
e, epsilon_0, a = sp.symbols("e epsilon_0 a", positive=True, real=True)
theta2, phi2 = sp.symbols("theta2 phi2", real=True)

psi1 = sp.symbols('psi1')

r1,r2         = sp.symbols('r1 r2',real=True,positive=True)
phi1,phi2     = sp.symbols('phi1 phi2',real=True)
theta1,theta2 = sp.symbols('theta1 theta2',real=True)


u = sp.symbols('u',real=True,positive=True)
Z_eff1     = sp.symbols('Z_eff1',real=True,positive=True)
Z_eff2     = sp.symbols('Z_eff2',real=True,positive=True)


Z         = sp.symbols('Z',real=True,positive=True)
E= sp.symbols('E')


psi = sp.Function('psi')



In [3]:
import sympy as sp
from sympy import symbols, Ynm
# 1. Define standard symbols and your function
r, theta, phi = sp.symbols('r theta phi', positive=True)

def attackSQRT(expr):
    #sympy struggles when sqrts are involved. 
    sqrt_map = {
        root: sp.sqrt(root.args[0].factor()) 
        for root in expr.find(sp.Pow) 
        if root.args[1] == sp.Rational(1, 2)
    }
    
    return expr.subs(sqrt_map)

def sphericalLaplacian(f,r,theta,phi):
    term_r = (1 / r**2) * sp.diff(r**2 * sp.diff(f, r), r)
    term_theta = (1 / (r**2 * sp.sin(theta))) * sp.diff(sp.sin(theta) * sp.diff(f, theta), theta)
    term_phi = (1 / (r**2 * sp.sin(theta)**2)) * sp.diff(f, phi, 2)
    laplacian = term_r + term_theta + term_phi
    sp.simplify(laplacian)
    return laplacian


def oneParticleIntegral(expr, r_var, theta_var, phi_var):
    # Assuming expr uses specific variables, or you can substitute them if needed:
    phiint   = sp.Integral(expr, (phi_var, 0, 2*sp.pi)).simplify()
    #print(phiint)
    thetaint = sp.Integral(phiint * sp.sin(theta_var), (theta_var, 0, sp.pi)).simplify()
    #print(thetaint)
    #The theta integral in the 2-electron integrals introduces sqrts 
    rint     = sp.Integral(attackSQRT(thetaint*r_var*r_var), (r_var, 0, sp.oo)).simplify()
    #print(rint)
    #print(rint)
    
    return rint

def twoParticleIntegral(expr):
    oneInt = oneParticleIntegral(expr,   r1, theta1, phi1)
    twoInt = oneParticleIntegral(oneInt, r2, theta2, phi2)
    return twoInt



In [4]:
Y = Ynm(1, 0, theta, phi)

In [5]:
Y

Ynm(1, 0, theta, phi)

In [6]:
#sphericalLaplacian(r*Y)

In [7]:
Ynm(0, 0, theta, phi).expand(func=True)**4

1/(16*pi**2)

In [8]:
def oneSorbital(r,theta,phi,Zeff):
    
    return 2 * sp.sqrt(Zeff**3) * sp.exp(-Zeff*r) *  Ynm(0, 0, theta, phi).expand(func=True)

In [9]:
oneSorbital(r2,theta2,phi2,Z_eff1)

Z_eff1**(3/2)*exp(-Z_eff1*r2)/sqrt(pi)

In [10]:
He_trial_GS = oneSorbital(r1,theta1,phi1,Z_eff1) * oneSorbital(r2,theta2,phi2,Z_eff2)  + oneSorbital(r1,theta1,phi1,Z_eff2) * oneSorbital(r2,theta2,phi2,Z_eff1) 

In [11]:
He_trial_GS

Z_eff1**(3/2)*Z_eff2**(3/2)*exp(-Z_eff1*r2)*exp(-Z_eff2*r1)/pi + Z_eff1**(3/2)*Z_eff2**(3/2)*exp(-Z_eff1*r1)*exp(-Z_eff2*r2)/pi

In [12]:
normFactor = 1/ sp.sqrt(twoParticleIntegral(He_trial_GS * He_trial_GS).simplify())

In [31]:
normFactor

sqrt(2)*(Z_eff1 + Z_eff2)**3/(2*sqrt(64*Z_eff1**3*Z_eff2**3 + (Z_eff1 + Z_eff2)**3*(Z_eff1**3 + 3*Z_eff1**2*Z_eff2 + 3*Z_eff1*Z_eff2**2 + Z_eff2**3)))

In [13]:
#He_trial_GS = normFactor * He_trial_GS 

In [14]:
He_trial_GS

Z_eff1**(3/2)*Z_eff2**(3/2)*exp(-Z_eff1*r2)*exp(-Z_eff2*r1)/pi + Z_eff1**(3/2)*Z_eff2**(3/2)*exp(-Z_eff1*r1)*exp(-Z_eff2*r2)/pi

In [15]:
#check normalization...
twoParticleIntegral(He_trial_GS * He_trial_GS).simplify()

2*(64*Z_eff1**3*Z_eff2**3 + (Z_eff1 + Z_eff2)**3*(Z_eff1**3 + 3*Z_eff1**2*Z_eff2 + 3*Z_eff1*Z_eff2**2 + Z_eff2**3))/(Z_eff1 + Z_eff2)**6

In [16]:
from sympy import S

frac = S(1)/2
KEOperator = (sphericalLaplacian(He_trial_GS, r1, theta1, phi1) + sphericalLaplacian(He_trial_GS, r2, theta2, phi2)).simplify() * (-frac)

In [17]:
KEOperator

-Z_eff1**(3/2)*Z_eff2**(3/2)*(r1*(-2*Z_eff1*exp(Z_eff1*r1 + Z_eff2*r2) - 2*Z_eff2*exp(Z_eff1*r2 + Z_eff2*r1) + r2*(Z_eff1**2*exp(Z_eff1*r1 + Z_eff2*r2) + Z_eff2**2*exp(Z_eff1*r2 + Z_eff2*r1))) + r2*(-2*Z_eff1*exp(Z_eff1*r2 + Z_eff2*r1) - 2*Z_eff2*exp(Z_eff1*r1 + Z_eff2*r2) + r1*(Z_eff1**2*exp(Z_eff1*r2 + Z_eff2*r1) + Z_eff2**2*exp(Z_eff1*r1 + Z_eff2*r2))))*exp(-Z_eff1*r1 - Z_eff1*r2 - Z_eff2*r1 - Z_eff2*r2)/(2*pi*r1*r2)

In [18]:
KEIntegral = twoParticleIntegral( KEOperator*He_trial_GS )
KEIntegral

-2*(-Z_eff1**5/2 - 3*Z_eff1**4*Z_eff2/2 - 2*Z_eff1**3*Z_eff2**2 - 2*Z_eff1**2*Z_eff2**3 - 16*Z_eff1**2*Z_eff2**3/(1 + Z_eff2/Z_eff1)**2 + 16*Z_eff1**2*Z_eff2**3/(1 + Z_eff2/Z_eff1)**3 - 3*Z_eff1*Z_eff2**4/2 - 16*Z_eff1*Z_eff2**4/(1 + Z_eff2/Z_eff1)**2 - 32*Z_eff1*Z_eff2**4/(1 + Z_eff2/Z_eff1)**3 - Z_eff2**5/2 + 16*Z_eff2**5/(1 + Z_eff2/Z_eff1)**3)/(Z_eff1 + Z_eff2)**3

In [19]:
twoParticleIntegral(1/r1 * He_trial_GS*He_trial_GS)

4*Z_eff1*Z_eff2*(Z_eff1**2/(4*Z_eff2) + 3*Z_eff1/4 + 3*Z_eff2/4 + Z_eff2**2/(4*Z_eff1) + 16*Z_eff2**2/(Z_eff1*(1 + Z_eff2/Z_eff1)**3))/(Z_eff1 + Z_eff2)**2

In [20]:
nuclearAttraction = -Z/r1 + -Z/r2
nuclearAttractionIntegral = twoParticleIntegral(nuclearAttraction*He_trial_GS*He_trial_GS)
nuclearAttractionIntegral

-4*Z*(Z_eff1**4/2 + 2*Z_eff1**3*Z_eff2 + 3*Z_eff1**2*Z_eff2**2 + 2*Z_eff1*Z_eff2**3 + 16*Z_eff1*Z_eff2**3/(1 + Z_eff2/Z_eff1)**2 + 16*Z_eff1*Z_eff2**3/(1 + Z_eff2/Z_eff1)**3 + Z_eff2**4/2 + 16*Z_eff2**4/(1 + Z_eff2/Z_eff1)**3)/(Z_eff1 + Z_eff2)**3

In [21]:
coulombRepulsion = 1/ (sp.sqrt(r1**2 + r2**2 - 2*r1*r2 * sp.cos(theta1)))
coulombRepulsionIntegral = twoParticleIntegral(coulombRepulsion*He_trial_GS*He_trial_GS)

In [22]:
coulombRepulsionIntegral

-4*(-Z_eff1**4/4 + Z_eff1**4/(4*(1 + Z_eff2/Z_eff1)**2) - Z_eff1**3*Z_eff2 + 3*Z_eff1**3*Z_eff2/(4*(1 + Z_eff2/Z_eff1)**2) + Z_eff1**3*Z_eff2/(4*(1 + Z_eff2/Z_eff1)**3) - 3*Z_eff1**2*Z_eff2**2/2 + 3*Z_eff1**2*Z_eff2**2/(4*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff1**2*Z_eff2**2/(4*(1 + Z_eff2/Z_eff1)**3) - Z_eff1*Z_eff2**3 - 23*Z_eff1*Z_eff2**3/(2*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff1*Z_eff2**3/(1 + Z_eff2/Z_eff1)**3 - Z_eff2**4/4 + 3*Z_eff2**4/(4*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff2**4/(1 + Z_eff2/Z_eff1)**3 + 3*Z_eff2**5/(4*Z_eff1*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff2**5/(4*Z_eff1*(1 + Z_eff2/Z_eff1)**3) + Z_eff2**6/(4*Z_eff1**2*(1 + Z_eff2/Z_eff1)**2) + Z_eff2**6/(4*Z_eff1**2*(1 + Z_eff2/Z_eff1)**3))/(Z_eff1**3 + 3*Z_eff1**2*Z_eff2 + 3*Z_eff1*Z_eff2**2 + Z_eff2**3)

In [23]:
totalEnergy = KEIntegral + nuclearAttractionIntegral + coulombRepulsionIntegral

In [24]:
totalEnergyNormed = totalEnergy / twoParticleIntegral(He_trial_GS * He_trial_GS).simplify() #normalize...

In [25]:
A = totalEnergy
B = twoParticleIntegral(He_trial_GS * He_trial_GS).simplify()

In [26]:
sp.diff(A,Z_eff1)

-4*Z*(2*Z_eff1**3 + 6*Z_eff1**2*Z_eff2 + 6*Z_eff1*Z_eff2**2 + 2*Z_eff2**3 + 16*Z_eff2**3/(1 + Z_eff2/Z_eff1)**2 + 16*Z_eff2**3/(1 + Z_eff2/Z_eff1)**3 + 32*Z_eff2**4/(Z_eff1*(1 + Z_eff2/Z_eff1)**3) + 48*Z_eff2**4/(Z_eff1*(1 + Z_eff2/Z_eff1)**4) + 48*Z_eff2**5/(Z_eff1**2*(1 + Z_eff2/Z_eff1)**4))/(Z_eff1 + Z_eff2)**3 + 12*Z*(Z_eff1**4/2 + 2*Z_eff1**3*Z_eff2 + 3*Z_eff1**2*Z_eff2**2 + 2*Z_eff1*Z_eff2**3 + 16*Z_eff1*Z_eff2**3/(1 + Z_eff2/Z_eff1)**2 + 16*Z_eff1*Z_eff2**3/(1 + Z_eff2/Z_eff1)**3 + Z_eff2**4/2 + 16*Z_eff2**4/(1 + Z_eff2/Z_eff1)**3)/(Z_eff1 + Z_eff2)**4 - 4*(-3*Z_eff1**2 - 6*Z_eff1*Z_eff2 - 3*Z_eff2**2)*(-Z_eff1**4/4 + Z_eff1**4/(4*(1 + Z_eff2/Z_eff1)**2) - Z_eff1**3*Z_eff2 + 3*Z_eff1**3*Z_eff2/(4*(1 + Z_eff2/Z_eff1)**2) + Z_eff1**3*Z_eff2/(4*(1 + Z_eff2/Z_eff1)**3) - 3*Z_eff1**2*Z_eff2**2/2 + 3*Z_eff1**2*Z_eff2**2/(4*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff1**2*Z_eff2**2/(4*(1 + Z_eff2/Z_eff1)**3) - Z_eff1*Z_eff2**3 - 23*Z_eff1*Z_eff2**3/(2*(1 + Z_eff2/Z_eff1)**2) + 3*Z_eff1*Z_eff2**3/

In [27]:
check = sp.lambdify((Z,Z_eff1,Z_eff2), totalEnergyNormed)

In [28]:
check(1,1.039,0.283) #which is bound.

-0.5133028427736092

In [29]:
def energy(params):
    a,b = params
    return check(1,a,b)
from scipy.optimize import minimize
res = minimize(energy, x0=(1,1),method='Nelder-Mead')
print(res.x)
print(res.fun)

[1.03926229 0.28319009]
-0.5133028843680565


In [30]:
def energy(params):
    a,b = params
    return check(2,a,b)
from scipy.optimize import minimize
res = minimize(energy, x0=(1,1),method='Nelder-Mead')
print(res.x)
print(res.fun)

[1.18850766 2.18316805]
-2.8756613310017793
